In [78]:
import json
import math
import os
from openai import OpenAI
from sentence_transformers import SentenceTransformer, CrossEncoder
from qdrant_client import QdrantClient, models
from qdrant_client.models import SparseTextEmbedding
from thefuzz import fuzz
from fastembed import SparseTextEmbedding

In [79]:
QDRANT_HOST = "localhost"
QDRANT_PORT = 6333

In [80]:
client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)

In [81]:
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

llm_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

OPENROUTER_MODEL = "openai/gpt-5.4"

In [82]:
def is_similar(text1, text2, threshold=80):
    return fuzz.partial_ratio(text1.strip(), text2.strip()) >= threshold

In [83]:
with open("final_golden_dataset.json", "r", encoding="utf-8") as f:
    golden_data = json.load(f)

In [84]:
reranker_model = CrossEncoder('BAAI/bge-reranker-v2-m3')

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 3176.81it/s]


In [85]:
def get_reranked_results(query, initial_results, k=5):
    points = initial_results.points
    pairs = [[query, res.payload['text']] for res in points]
    scores = reranker_model.predict(pairs)
    scored_results = sorted(zip(points, scores), key=lambda x: x[1], reverse=True)

    return [res[0] for res in scored_results[:k]]

In [86]:
def evaluate_retrieval_metrics(search_results, golden_context, k=5):
    hit = 0
    reciprocal_rank = 0
    dcg = 0
    found_golden_indices = set()
    
    for rank, result in enumerate(search_results[:k], start=1):
        for i, golden_text in enumerate(golden_context):
            if i not in found_golden_indices:
                if is_similar(golden_text, result):
                    if hit == 0:
                        hit = 1
                        reciprocal_rank = 1 / rank
                    
                    found_golden_indices.add(i)
                    dcg += 1 / math.log2(rank + 1)
                    break

    relevant_found_count = len(found_golden_indices)
    recall = relevant_found_count / len(golden_context) if len(golden_context) > 0 else 0
    
    idcg = sum(1 / math.log2(i + 1) for i in range(1, min(len(golden_context), k) + 1))
    ndcg = dcg / idcg if idcg > 0 else 0
    
    return hit, reciprocal_rank, recall, ndcg

<hr>

## Evaluate baseline model

In [87]:
def get_metrics(model, collection, e5=False, openai_client=None):
    total_hit = 0
    total_mrr = 0
    total_recall = 0
    total_ndcg = 0

    total_hit_reranked = 0
    total_mrr_reranked = 0
    total_recall_reranked = 0
    total_ndcg_reranked = 0

    for item in golden_data:
        if openai_client:
            response = openai_client.embeddings.create(
                input=[item['input'].replace("\n", " ")],
                model="text-embedding-3-large"
            )
            query = response.data[0].embedding
        elif e5:
            query = model.encode("query: " + item['input']).tolist()
        else:
            query=model.encode(item['input']).tolist()
        results = client.query_points(
            collection_name=collection,
            prefetch=[
                models.Prefetch(
                    query=query,
                    using="default",
                    limit=20
                )
            ],
            query=models.FusionQuery(fusion=models.Fusion.RRF),
            limit=20
        )
        found_texts = [res.payload['text'] for res in results.points]
        
        h, m, r, n = evaluate_retrieval_metrics(found_texts, item['retrieval_context'])
        total_hit += h
        total_mrr += m
        total_recall += r
        total_ndcg += n

        reranked_objects = get_reranked_results(item['input'], results)
        reranked_texts = [res.payload['text'] for res in reranked_objects]

        h_rerank, m_rerank, r_rerank, n_rerank = evaluate_retrieval_metrics(reranked_texts, item['retrieval_context'])
        total_hit_reranked += h_rerank
        total_mrr_reranked += m_rerank
        total_recall_reranked += r_rerank
        total_ndcg_reranked += n_rerank

    return total_hit, total_mrr, total_recall, total_ndcg, \
            total_hit_reranked, total_mrr_reranked, total_recall_reranked, total_ndcg_reranked

In [88]:
MODEL_BASELINE = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
COLLECTION_BASELINE = "ucu_documents_baseline"

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2057.58it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [89]:
total_hit_baseline, total_mrr_baseline, total_recall_baseline, total_ndcg_baseline, \
total_hit_reranked_baseline, total_mrr_reranked_baseline, total_recall_reranked_baseline, total_ndcg_reranked_baseline = get_metrics(MODEL_BASELINE, COLLECTION_BASELINE)

In [90]:
n = len(golden_data)

In [91]:
print("Results for baseline model:")
print(f"Hit Rate @ 5: {round(total_hit_baseline/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_baseline/n, 2)}")
print(f"Recall @ 5: {round(total_recall_baseline/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_baseline/n, 2)}")

Results for baseline model:
Hit Rate @ 5: 0.5
MRR @ 5: 0.31
Recall @ 5: 0.42
NDCG @ 5: 0.32


<hr>

## Evaluate baseline model with clever chunking

In [92]:
COLLECTION_BASELINE_RECURSIVE_CHAR = "ucu_documents_baseline_recursive_char"

In [93]:
total_hit_baseline_2, total_mrr_baseline_2, total_recall_baseline_2, total_ndcg_baseline_2, \
total_hit_reranked_baseline_2, total_mrr_reranked_baseline_2, total_recall_reranked_baseline_2, total_ndcg_reranked_baseline_2 = get_metrics(MODEL_BASELINE, COLLECTION_BASELINE_RECURSIVE_CHAR)

In [94]:
print("Results for baseline model (different chunking):")
print(f"Hit Rate @ 5: {round(total_hit_baseline_2/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_baseline_2/n, 2)}")
print(f"Recall @ 5: {round(total_recall_baseline_2/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_baseline_2/n, 2)}")

Results for baseline model (different chunking):
Hit Rate @ 5: 0.45
MRR @ 5: 0.3
Recall @ 5: 0.42
NDCG @ 5: 0.33


Let`s see if the model performs better with reranker

In [95]:
print("Results for baseline model (different chunking + reranker):")
print(f"Hit Rate @ 5: {round(total_hit_reranked_baseline_2/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_baseline_2/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_baseline_2/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_baseline_2/n, 2)}")

Results for baseline model (different chunking + reranker):
Hit Rate @ 5: 0.7
MRR @ 5: 0.64
Recall @ 5: 0.65
NDCG @ 5: 0.61


<hr>

## Evaluate baseline model with addition of sparse vectors - Hybrid Search

In [96]:
MODEL_SPARSE = SparseTextEmbedding(model_name="Qdrant/bm25")
COLLECTION_BASELINE_HYBRID = "ucu_documents_baseline_hybrid"

In [97]:
def get_metrics_sparse(model, collection, sparse_model, e5=False, openai_client=None):
    total_hit = 0
    total_mrr = 0
    total_recall = 0
    total_ndcg = 0

    total_hit_reranked = 0
    total_mrr_reranked = 0
    total_recall_reranked = 0
    total_ndcg_reranked = 0

    for item in golden_data:
        if openai_client:
            response = openai_client.embeddings.create(
                input=[item['input'].replace("\n", " ")],
                model="text-embedding-3-large"
            )
            dense_vector = response.data[0].embedding
        elif e5:
            dense_vector = model.encode("query: " + item['input']).tolist()
        else:
            dense_vector = model.encode(item['input']).tolist()
        sparse_emb = list(sparse_model.embed([item['input']]))[0]
            
        results = client.query_points(
            collection_name=collection,
            prefetch=[
                models.Prefetch(
                    query=dense_vector,
                    using="default",
                    limit=20
                ),
                models.Prefetch(
                    query=models.SparseVector(
                    indices=sparse_emb.indices.tolist(),
                    values=sparse_emb.values.tolist()
                ),
                    using="text_sparse",
                    limit=20
                ),
            ],
            query=models.FusionQuery(fusion=models.Fusion.RRF),
            limit=20
        )
        found_texts = [res.payload['text'] for res in results.points]
        
        h, m, r, n = evaluate_retrieval_metrics(found_texts, item['retrieval_context'])
        total_hit += h
        total_mrr += m
        total_recall += r
        total_ndcg += n

        reranked_objects = get_reranked_results(item['input'], results)
        reranked_texts = [res.payload['text'] for res in reranked_objects]

        h_rerank, m_rerank, r_rerank, n_rerank = evaluate_retrieval_metrics(reranked_texts, item['retrieval_context'])
        total_hit_reranked += h_rerank
        total_mrr_reranked += m_rerank
        total_recall_reranked += r_rerank
        total_ndcg_reranked += n_rerank

    return total_hit, total_mrr, total_recall, total_ndcg, \
            total_hit_reranked, total_mrr_reranked, total_recall_reranked, total_ndcg_reranked

In [98]:
total_hit_baseline_sparse, total_mrr_baseline_sparse, total_recall_baseline_sparse, total_ndcg_baseline_sparse, \
total_hit_reranked_baseline_sparse, total_mrr_reranked_baseline_sparse, total_recall_reranked_baseline_sparse, total_ndcg_reranked_baseline_sparse = get_metrics_sparse(MODEL_BASELINE, COLLECTION_BASELINE_HYBRID, MODEL_SPARSE)

In [99]:
print("Results for baseline model + sparse vectors added:")
print(f"Hit Rate @ 5: {round(total_hit_reranked_baseline_sparse/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_baseline_sparse/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_baseline_sparse/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_baseline_sparse/n, 2)}")

Results for baseline model + sparse vectors added:
Hit Rate @ 5: 0.7
MRR @ 5: 0.66
Recall @ 5: 0.63
NDCG @ 5: 0.62


<hr>

## Evaluate baseline model with HyDE Search

In [100]:
def generate_hypothetical_answer(query, gemini_model):
    prompt = f"""
    Ти — асистент адміністрації Українського Католицького Університету (УКУ).

    Напиши логічну та лаконічну відповідь на запитання користувача.
    Використовуй офіційний стиль і термінологію, притаманну наказам та положенням УКУ.

    Сформулюй відповідь так, щоб вона могла містити:
    - ключові терміни та формулювання, які використовуються в офіційних документах
    - альтернативні або близькі за змістом формулювання (без значного збільшення обсягу)

    Не додавай зайвої інформації, але можеш узагальнювати формулювання для кращого покриття теми.

    Запитання: {query}
    """
    response = llm_client.chat.completions.create(
        model=gemini_model,
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": query}
            ],
            temperature=0.1
    )
    return response.choices[0].message.content

In [142]:
def get_metrics_hyde(model, collection, openrouter_model, e5=False, openai_client=None, sparse_model=None):
    total_hit = 0
    total_mrr = 0
    total_recall = 0
    total_ndcg = 0

    total_hit_reranked = 0
    total_mrr_reranked = 0
    total_recall_reranked = 0
    total_ndcg_reranked = 0

    for item in golden_data:
        hyde_doc = generate_hypothetical_answer(item['input'], openrouter_model)
        if openai_client:
            response = openai_client.embeddings.create(
                input=[item['input'].replace("\n", " ")],
                model="text-embedding-3-large"
            )
            query_vector = response.data[0].embedding
        elif e5:
            query_vector = model.encode("query: " + hyde_doc).tolist()
        else:
            query_vector = model.encode(hyde_doc).tolist()

        if sparse_model:
            sparse_emb = list(sparse_model.embed([item['input']]))[0]
            results = client.query_points(
                collection_name=collection,
                prefetch=[
                    models.Prefetch(
                        query=query_vector,
                        using="default",
                        limit=20
                    ),
                    models.Prefetch(
                        query=models.SparseVector(
                        indices=sparse_emb.indices.tolist(),
                        values=sparse_emb.values.tolist()
                    ),
                        using="text_sparse",
                        limit=20
                    ),
                ],
                query=models.FusionQuery(fusion=models.Fusion.RRF),
                limit=20
            ) 
        else:
            results = client.query_points(
                collection_name=collection,
                prefetch=[
                    models.Prefetch(
                        query=query_vector,
                        using="default",
                        limit=20
                    )
                ],
                query=models.FusionQuery(fusion=models.Fusion.RRF),
                limit=20
            )
        found_texts = [res.payload['text'] for res in results.points]
        
        h, m, r, n = evaluate_retrieval_metrics(found_texts, item['retrieval_context'])
        total_hit += h
        total_mrr += m
        total_recall += r
        total_ndcg += n

        reranked_objects = get_reranked_results(item['input'], results)
        reranked_texts = [res.payload['text'] for res in reranked_objects]

        h_rerank, m_rerank, r_rerank, n_rerank = evaluate_retrieval_metrics(reranked_texts, item['retrieval_context'])
        total_hit_reranked += h_rerank
        total_mrr_reranked += m_rerank
        total_recall_reranked += r_rerank
        total_ndcg_reranked += n_rerank

    return total_hit, total_mrr, total_recall, total_ndcg, \
            total_hit_reranked, total_mrr_reranked, total_recall_reranked, total_ndcg_reranked

In [102]:
total_hit_baseline_hyde, total_mrr_baseline_hyde, total_recall_baseline_hyde, total_ndcg_baseline_hyde, \
total_hit_reranked_baseline_hyde, total_mrr_reranked_baseline_hyde, total_recall_reranked_baseline_hyde, total_ndcg_reranked_baseline_hyde = get_metrics_hyde(MODEL_BASELINE, COLLECTION_BASELINE_RECURSIVE_CHAR, OPENROUTER_MODEL)

In [103]:
print("Results for baseline model + HyDE Search:")
print(f"Hit Rate @ 5: {round(total_hit_reranked_baseline_hyde/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_baseline_hyde/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_baseline_hyde/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_baseline_hyde/n, 2)}")

Results for baseline model + HyDE Search:
Hit Rate @ 5: 0.75
MRR @ 5: 0.69
Recall @ 5: 0.65
NDCG @ 5: 0.62


<hr>

## Evaluate E5 small model (Naive)

In [104]:
MODEL_E5_SMALL = SentenceTransformer('intfloat/multilingual-e5-small')
COLLECTION_E5_SMALL = "ucu_documents_e5_small"

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3362.88it/s]
BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [105]:
total_hit_e5_small, total_mrr_e5_small, total_recall_e5_small, total_ndcg_e5_small, \
total_hit_reranked_e5_small, total_mrr_reranked_e5_small, total_recall_reranked_e5_small, total_ndcg_reranked_e5_small = get_metrics(MODEL_E5_SMALL, COLLECTION_E5_SMALL, e5=True)

In [106]:
print("Results for E5 small model:")
print(f"Hit Rate @ 5: {round(total_hit_reranked_e5_small/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_e5_small/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_e5_small/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_e5_small/n, 2)}")

Results for E5 small model:
Hit Rate @ 5: 0.8
MRR @ 5: 0.76
Recall @ 5: 0.7
NDCG @ 5: 0.69


<hr>

## Evaluate E5 small model (Hybrid)

In [107]:
total_hit_e5_small_sparse, total_mrr_e5_small_sparse, total_recall_e5_small_sparse, total_ndcg_e5_small_sparse, \
total_hit_reranked_e5_small_sparse, total_mrr_reranked_e5_small_sparse, total_recall_reranked_e5_small_sparse, total_ndcg_reranked_e5_small_sparse = get_metrics_sparse(MODEL_E5_SMALL, COLLECTION_E5_SMALL, sparse_model=MODEL_SPARSE, e5=True)

In [108]:
print("Results for E5 small model + sparse vectors:")
print(f"Hit Rate @ 5: {round(total_hit_reranked_e5_small_sparse/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_e5_small_sparse/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_e5_small_sparse/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_e5_small_sparse/n, 2)}")

Results for E5 small model + sparse vectors:
Hit Rate @ 5: 0.8
MRR @ 5: 0.76
Recall @ 5: 0.68
NDCG @ 5: 0.68


<hr>

## Evaluate E5 small model (HyDE)

In [109]:
total_hit_e5_small_hyde, total_mrr_e5_small_hyde, total_recall_e5_small_hyde, total_ndcg_e5_small_hyde, \
total_hit_reranked_e5_small_hyde, total_mrr_reranked_e5_small_hyde, total_recall_reranked_e5_small_hyde, total_ndcg_reranked_e5_small_hyde = get_metrics_hyde(MODEL_E5_SMALL, COLLECTION_E5_SMALL, OPENROUTER_MODEL)

In [110]:
print("Results for E5 small model + HyDE Search:")
print(f"Hit Rate @ 5: {round(total_hit_reranked_e5_small_hyde/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_e5_small_hyde/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_e5_small_hyde/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_e5_small_hyde/n, 2)}")

Results for E5 small model + HyDE Search:
Hit Rate @ 5: 0.7
MRR @ 5: 0.64
Recall @ 5: 0.57
NDCG @ 5: 0.56


<hr>

## Evaluate E5 base model (Naive)

In [113]:
MODEL_E5_BASE = SentenceTransformer('intfloat/multilingual-e5-base')
COLLECTION_E5_BASE = "ucu_documents_e5_base"

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2014.65it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [114]:
total_hit_e5_base, total_mrr_e5_base, total_recall_e5_base, total_ndcg_e5_base, \
total_hit_reranked_e5_base, total_mrr_reranked_e5_base, total_recall_reranked_e5_base, total_ndcg_reranked_e5_base = get_metrics(MODEL_E5_BASE, COLLECTION_E5_BASE, e5=True)

In [115]:
print("Results for E5 base model:")
print(f"Hit Rate @ 5: {round(total_hit_reranked_e5_base/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_e5_base/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_e5_base/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_e5_base/n, 2)}")

Results for E5 base model:
Hit Rate @ 5: 0.85
MRR @ 5: 0.76
Recall @ 5: 0.69
NDCG @ 5: 0.67


<hr>

## Evaluate E5 base model (Hybrid)

In [116]:
total_hit_e5_base_sparse, total_mrr_e5_base_sparse, total_recall_e5_base_sparse, total_ndcg_e5_base_sparse, \
total_hit_reranked_e5_base_sparse, total_mrr_reranked_e5_base_sparse, total_recall_reranked_e5_base_sparse, total_ndcg_reranked_e5_base_sparse = get_metrics_sparse(MODEL_E5_BASE, COLLECTION_E5_BASE, sparse_model=MODEL_SPARSE, e5=True)

In [117]:
print("Results for E5 base model + sparse vectors:")
print(f"Hit Rate @ 5: {round(total_hit_reranked_e5_base_sparse/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_e5_base_sparse/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_e5_base_sparse/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_e5_base_sparse/n, 2)}")

Results for E5 base model + sparse vectors:
Hit Rate @ 5: 0.85
MRR @ 5: 0.78
Recall @ 5: 0.73
NDCG @ 5: 0.71


<hr>

## Evaluate E5 base model (HyDE)

In [118]:
total_hit_e5_base_hyde, total_mrr_e5_base_hyde, total_recall_e5_base_hyde, total_ndcg_e5_base_hyde, \
total_hit_reranked_e5_base_hyde, total_mrr_reranked_e5_base_hyde, total_recall_reranked_e5_base_hyde, total_ndcg_reranked_e5_base_hyde = get_metrics_hyde(MODEL_E5_BASE, COLLECTION_E5_BASE, OPENROUTER_MODEL)

In [119]:
print("Results for E5 base model + HyDE Search:")
print(f"Hit Rate @ 5: {round(total_hit_reranked_e5_base_hyde/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_e5_base_hyde/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_e5_base_hyde/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_e5_base_hyde/n, 2)}")

Results for E5 base model + HyDE Search:
Hit Rate @ 5: 0.6
MRR @ 5: 0.56
Recall @ 5: 0.44
NDCG @ 5: 0.45


<HR>

## Evaluate E5 large model (Naive)

In [120]:
MODEL_E5_LARGE = SentenceTransformer('intfloat/multilingual-e5-large')
COLLECTION_E5_LARGE = "ucu_documents_e5_large"

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3209.06it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [121]:
total_hit_e5_large, total_mrr_e5_large, total_recall_e5_large, total_ndcg_e5_large, \
total_hit_reranked_e5_large, total_mrr_reranked_e5_large, total_recall_reranked_e5_large, total_ndcg_reranked_e5_large = get_metrics(MODEL_E5_LARGE, COLLECTION_E5_LARGE, e5=True)

In [122]:
print("Results for E5 large model:")
print(f"Hit Rate @ 5: {round(total_hit_reranked_e5_large/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_e5_large/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_e5_large/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_e5_large/n, 2)}")

Results for E5 large model:
Hit Rate @ 5: 0.95
MRR @ 5: 0.86
Recall @ 5: 0.79
NDCG @ 5: 0.77


<hr>

## Evaluate E5 large model (Hybrid)

In [123]:
total_hit_e5_large_sparse, total_mrr_e5_large_sparse, total_recall_e5_large_sparse, total_ndcg_e5_large_sparse, \
total_hit_reranked_e5_large_sparse, total_mrr_reranked_e5_large_sparse, total_recall_reranked_e5_large_sparse, total_ndcg_reranked_e5_large_sparse = get_metrics_sparse(MODEL_E5_LARGE, COLLECTION_E5_LARGE, sparse_model=MODEL_SPARSE, e5=True)

In [124]:
print("Results for E5 large model + sparse vectors:")
print(f"Hit Rate @ 5: {round(total_hit_reranked_e5_large_sparse/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_e5_large_sparse/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_e5_large_sparse/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_e5_large_sparse/n, 2)}")

Results for E5 large model + sparse vectors:
Hit Rate @ 5: 0.9
MRR @ 5: 0.84
Recall @ 5: 0.78
NDCG @ 5: 0.77


<hr>

## Evaluate E5 large model (HyDE)

In [125]:
total_hit_e5_large_hyde, total_mrr_e5_large_hyde, total_recall_e5_large_hyde, total_ndcg_e5_large_hyde, \
total_hit_reranked_e5_large_hyde, total_mrr_reranked_e5_large_hyde, total_recall_reranked_e5_large_hyde, total_ndcg_reranked_e5_large_hyde = get_metrics_hyde(MODEL_E5_LARGE, COLLECTION_E5_LARGE, OPENROUTER_MODEL, e5=True)

In [126]:
print("Results for E5 large model + HyDE Search:")
print(f"Hit Rate @ 5: {round(total_hit_reranked_e5_large_hyde/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_e5_large_hyde/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_e5_large_hyde/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_e5_large_hyde/n, 2)}")

Results for E5 large model + HyDE Search:
Hit Rate @ 5: 0.7
MRR @ 5: 0.66
Recall @ 5: 0.56
NDCG @ 5: 0.56


<hr>

## Evaluate Ukrainian model (Naive)

In [127]:
MODEL_UKR = SentenceTransformer('lang-uk/ukr-paraphrase-multilingual-mpnet-base')
COLLECTION_UKR = "ucu_documents_ukr"

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 19827.69it/s]
XLMRobertaModel LOAD REPORT from: lang-uk/ukr-paraphrase-multilingual-mpnet-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [128]:
total_hit_ukr, total_mrr_ukr, total_recall_ukr, total_ndcg_ukr, \
total_hit_reranked_ukr, total_mrr_reranked_ukr, total_recall_reranked_ukr, total_ndcg_reranked_ukr = get_metrics(MODEL_UKR, COLLECTION_UKR)

In [129]:
print("Results for Ukrainian model:")
print(f"Hit Rate @ 5: {round(total_hit_reranked_ukr/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_ukr/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_ukr/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_ukr/n, 2)}")

Results for Ukrainian model:
Hit Rate @ 5: 0.6
MRR @ 5: 0.53
Recall @ 5: 0.55
NDCG @ 5: 0.5


<hr>

## Evaluate Ukrainian model (Hybrid)

In [130]:
total_hit_ukr_sparse, total_mrr_ukr_sparse, total_recall_ukr_sparse, total_ndcg_ukr_sparse, \
total_hit_reranked_ukr_sparse, total_mrr_reranked_ukr_sparse, total_recall_reranked_ukr_sparse, total_ndcg_reranked_ukr_sparse = get_metrics_sparse(MODEL_UKR, COLLECTION_UKR, sparse_model=MODEL_SPARSE)

In [131]:
print("Results for Ukrainian model + sparse vectors:")
print(f"Hit Rate @ 5: {round(total_hit_reranked_ukr_sparse/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_ukr_sparse/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_ukr_sparse/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_ukr_sparse/n, 2)}")

Results for Ukrainian model + sparse vectors:
Hit Rate @ 5: 0.65
MRR @ 5: 0.61
Recall @ 5: 0.61
NDCG @ 5: 0.59


<HR>

## Evaluate Ukrainian model (HyDE)

In [132]:
total_hit_ukr_hyde, total_mrr_ukr_hyde, total_recall_ukr_hyde, total_ndcg_ukr_hyde, \
total_hit_reranked_ukr_hyde, total_mrr_reranked_ukr_hyde, total_recall_reranked_ukr_hyde, total_ndcg_reranked_ukr_hyde = get_metrics_hyde(MODEL_UKR, COLLECTION_UKR, OPENROUTER_MODEL)

In [133]:
print("Results for Ukrainian model + HyDE Search:")
print(f"Hit Rate @ 5: {round(total_hit_reranked_ukr_hyde/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_ukr_hyde/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_ukr_hyde/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_ukr_hyde/n, 2)}")

Results for Ukrainian model + HyDE Search:
Hit Rate @ 5: 0.7
MRR @ 5: 0.64
Recall @ 5: 0.6
NDCG @ 5: 0.58


<hr>

## Evaluate OpenAI model (Naive)

In [134]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
openai_client = OpenAI(api_key=OPENAI_API_KEY)
COLLECTION_OPENAI = "ucu_documents_openai"

In [135]:
total_hit_openai, total_mrr_openai, total_recall_openai, total_ndcg_openai, \
total_hit_reranked_openai, total_mrr_reranked_openai, total_recall_reranked_openai, total_ndcg_reranked_openai = get_metrics(None, COLLECTION_OPENAI, openai_client=openai_client)

In [136]:
print("Results for OpenAI model:")
print(f"Hit Rate @ 5: {round(total_hit_reranked_openai/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_openai/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_openai/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_openai/n, 2)}")

Results for OpenAI model:
Hit Rate @ 5: 0.9
MRR @ 5: 0.81
Recall @ 5: 0.76
NDCG @ 5: 0.74


<hr>

## Evaluate OpenAI model (Hybrid)

In [137]:
total_hit_openai_sparse, total_mrr_openai_sparse, total_recall_openai_sparse, total_ndcg_openai_sparse, \
total_hit_reranked_openai_sparse, total_mrr_reranked_openai_sparse, total_recall_reranked_openai_sparse, total_ndcg_reranked_openai_sparse = get_metrics_sparse(None, COLLECTION_OPENAI, sparse_model=MODEL_SPARSE, openai_client=openai_client)

In [138]:
print("Results for OpenAI model + sparse vectors:")
print(f"Hit Rate @ 5: {round(total_hit_reranked_openai_sparse/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_openai_sparse/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_openai_sparse/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_openai_sparse/n, 2)}")

Results for OpenAI model + sparse vectors:
Hit Rate @ 5: 0.95
MRR @ 5: 0.89
Recall @ 5: 0.81
NDCG @ 5: 0.8


<hr>

## Evaluate OpenAI model (HyDE)

In [139]:
total_hit_openai_hyde, total_mrr_openai_hyde, total_recall_openai_hyde, total_ndcg_openai_hyde, \
total_hit_reranked_openai_hyde, total_mrr_reranked_openai_hyde, total_recall_reranked_openai_hyde, total_ndcg_reranked_openai_hyde = get_metrics_hyde(None, COLLECTION_OPENAI, OPENROUTER_MODEL, openai_client=openai_client)

In [140]:
print("Results for OpenAI model + HyDE Search:")
print(f"Hit Rate @ 5: {round(total_hit_reranked_openai_hyde/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_openai_hyde/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_openai_hyde/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_openai_hyde/n, 2)}")

Results for OpenAI model + HyDE Search:
Hit Rate @ 5: 0.9
MRR @ 5: 0.81
Recall @ 5: 0.76
NDCG @ 5: 0.74
